# Bouwen met de Meta Familie Modellen 

## Introductie 

Deze les behandelt: 

- Verkenning van de twee belangrijkste Meta familie modellen - Llama 3.1 en Llama 3.2, plus de momenteel beschikbare alternatieven die in de codevoorbeelden worden gebruikt (Llama 3.3 en Llama 4) 
- Begrip van de gebruikssituaties en scenario's voor elk model 
- Codevoorbeeld om de unieke functies van elk model te laten zien 


## De Meta Familie van Modellen 

In deze les zullen we 2 modellen van de Meta familie of "Llama Herd" verkennen - Llama 3.1 en Llama 3.2. De uitvoerbare codevoorbeelden gebruiken Llama 3.3 en Llama 4, die momenteel beschikbare alternatieven zijn in de Microsoft Foundry Models catalogus. 

Deze modellen komen in verschillende varianten en zijn beschikbaar in de [Microsoft Foundry Models catalogus](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Opmerking:** GitHub Models wordt met eind juli 2026 uitgefaseerd. Hier vindt u meer details over het gebruik van [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) om te prototypen met AI modellen.

> **Opmerking:** De specifieke modelversies waartegen deze les oorspronkelijk was geschreven (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) zijn inmiddels uitgefaseerd uit de Microsoft Foundry Models catalogus. De codevoorbeelden hieronder gebruiken `Llama-3.3-70B-Instruct` en `Llama-4-Scout-17B-16E-Instruct` als momenteel beschikbare alternatieven in de Microsoft Foundry Models catalogus, dus ze draaien nog steeds vandaag.

Modelvarianten (zoals oorspronkelijk behandeld in deze les): 
- Llama 3.1 - 70B Instruct (uitgefaseerd) 
- Llama 3.1 - 405B Instruct (uitgefaseerd) 
- Llama 3.2 - 11B Vision Instruct (uitgefaseerd) 
- Llama 3.2 - 90B Vision Instruct (uitgefaseerd) 

Momenteel beschikbare alternatieven gebruikt in de codevoorbeelden: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Opmerking: De hierboven besproken Llama 3 varianten worden voor historische context behouden en zijn niet langer beschikbaar in Microsoft Foundry Models.*



## Llama 3.1 

Met 405 miljard parameters valt Llama 3.1 in de categorie open source LLM. 

Het model is een upgrade ten opzichte van de eerdere release Llama 3 door te bieden: 

- Groter contextvenster - 128k tokens versus 8k tokens 
- Groter Max Output Tokens - 4096 versus 2048 
- Betere meertalige ondersteuning - door toename van trainings-tokens 

Dit stelt Llama 3.1 in staat om complexere use cases aan te kunnen bij het bouwen van GenAI-applicaties, waaronder: 
- Native Functie-aanroep - de mogelijkheid om externe tools en functies buiten de LLM-workflow aan te roepen
- Betere RAG-prestaties - dankzij het grotere contextvenster 
- Synthese Data Generatie - de mogelijkheid om effectieve data te creëren voor taken zoals fine-tuning 



### Native Function Calling 

Llama 3.1 is bijgesteld om effectiever te zijn in het aanroepen van functies of tools. Het heeft ook twee ingebouwde tools die het model kan herkennen als nodig om te gebruiken op basis van de prompt van de gebruiker. Deze tools zijn: 

- **Brave Search** - Kan worden gebruikt om actuele informatie te krijgen zoals het weer door een webzoekopdracht uit te voeren 
- **Wolfram Alpha** - Kan worden gebruikt voor complexere wiskundige berekeningen zodat je je eigen functies niet hoeft te schrijven. 

Je kunt ook je eigen aangepaste tools maken die het LLM kan aanroepen. 

In het onderstaande codevoorbeeld: 

- Definiëren we de beschikbare tools (brave_search, wolfram_alpha) in de systeemprompt. 
- Verstuur je een gebruikersprompt die vraagt naar het weer in een bepaalde stad. 
- Zal het LLM reageren met een tool-aanroep naar de Brave Search-tool die er zo uitziet `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Opmerking: Dit voorbeeld roept alleen de tool aan, als je de resultaten wilt krijgen, moet je een gratis account aanmaken op de Brave API-pagina en de functie zelf definiëren` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Ondanks dat het een LLM is, heeft Llama 3.1 een beperking in multimodaliteit. Dat wil zeggen, het kunnen gebruiken van verschillende soorten input zoals afbeeldingen als prompts en het geven van antwoorden. Deze functie is één van de belangrijkste kenmerken van Llama 3.2. Deze functies omvatten ook: 

- Multimodaliteit - heeft de mogelijkheid om zowel tekst- als afbeeldingsprompts te evalueren 
- Variaties in kleine tot middelgrote omvang (11B en 90B) - dit biedt flexibele implementatieopties, 
- Alleen-tekst variaties (1B en 3B) - dit maakt het mogelijk om het model op edge / mobiele apparaten te implementeren en zorgt voor lage latency 

De multimodale ondersteuning vertegenwoordigt een grote stap in de wereld van open source modellen. Het onderstaande codevoorbeeld neemt zowel een afbeelding als tekstprompt om een analyse van de afbeelding te krijgen. Omdat Llama 3.2 90B Vision Instruct is stopgezet, gebruikt het voorbeeld `Llama-4-Scout-17B-16E-Instruct`, een momenteel beschikbare native multimodale alternatief. 

### Multimodale ondersteuning met Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Het leren stopt hier niet, vervolg de reis

Na het voltooien van deze les, bekijk onze [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) om je kennis van Generative AI verder te vergroten!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
